# Cross-Night Emotion Alluvial Analysis

This notebook shows how to:

1. Call the DreamForge API multiple times to simulate several nights.
2. Collect dream segments into a tabular format.
3. Build an emotion-by-night Sankey/alluvial visualization.

It assumes the API is running at `http://localhost:8000` (e.g., via `docker-compose up`).


## 1. Imports and configuration

Make sure you have `requests`, `pandas`, and `plotly` installed in your environment.


In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go

API_BASE = "http://localhost:8000"


## 2. Helper to run multiple nights via the API

We use the `/api/simulation/night` endpoint to generate `num_nights` nights with shared parameters.


In [ ]:
def run_nights(num_nights: int = 5, duration_hours: float = 8.0, dt_minutes: float = 0.5, ssri_strength: float = 1.0, stress_level: float = 0.2):
    nights = []
    for i in range(num_nights):
        body = {
            "duration_hours": duration_hours,
            "dt_minutes": dt_minutes,
            "ssri_strength": ssri_strength,
            "stress_level": stress_level,
        }
        resp = requests.post(f"{API_BASE}/api/simulation/night", json=body)
        resp.raise_for_status()
        nights.append(resp.json())
    return nights


## 3. Run a small multi-night batch

Adjust the parameters below as needed.


In [ ]:
nights_raw = run_nights(num_nights=5, duration_hours=8.0, dt_minutes=0.5, ssri_strength=1.0, stress_level=0.2)
len(nights_raw)


## 4. Flatten segments into a DataFrame

We build a tabular representation with one row per dream segment, including night index, segment index, dominant emotion, and basic metrics.


In [ ]:
rows = []
for night_idx, night in enumerate(nights_raw):
    for seg_idx, seg in enumerate(night["segments"]):
        rows.append(
            {
                "night_index": night_idx,
                "segment_index": seg_idx,
                "stage": seg["stage"],
                "dominant_emotion": seg["dominant_emotion"],
                "bizarreness_score": seg["bizarreness_score"],
                "lucidity_probability": seg["lucidity_probability"],
            }
        )
segments_df = pd.DataFrame(rows)
segments_df.head()


## 5. Build an emotion-by-night Sankey/alluvial view

Here we aggregate how often each emotion appears in each night and build a Sankey diagram with:

- Sources: Night 1, Night 2, ...
- Targets: emotion categories (joy, fear, neutral, etc.).

This is a simple way to visualize how emotional tone is distributed across nights.


In [ ]:
if segments_df.empty:
    raise RuntimeError("No segments generated; check API status and parameters.")

emotion_counts = (
    segments_df.groupby(["night_index", "dominant_emotion"])
    .size()
    .reset_index(name="count")
)

nights = sorted(segments_df["night_index"].unique())
emotions = sorted(segments_df["dominant_emotion"].unique())

node_labels = [f"Night {i+1}" for i in nights] + emotions
night_to_node = {i: idx for idx, i in enumerate(nights)}
emotion_to_node = {emo: len(nights) + idx for idx, emo in enumerate(emotions)}

sources = []
targets = []
values = []
for _, row in emotion_counts.iterrows():
    sources.append(night_to_node[row["night_index"]])
    targets.append(emotion_to_node[row["dominant_emotion"]])
    values.append(int(row["count"]))

fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=node_labels, pad=20, thickness=20),
            link=dict(source=sources, target=targets, value=values),
        )
    ]
)
fig.update_layout(title_text="Emotion distribution across nights", font_size=12)
fig.show()


## 6. Extending to themes

If you expose memory fragment labels or tags via the API, a similar approach can be used to build a night-by-theme Sankey/alluvial diagram:

- Flatten segments with their associated memory IDs and tags.
- Aggregate counts per (night, theme) pair.
- Reuse the same Sankey construction pattern as above.

This notebook focuses on emotions, but the structure is directly reusable for richer theme-level analysis once the API exposes the necessary fields.
